# Run Email Assistant via SDK

This notebook loads the compiled agent from the project and runs it via `invoke()` (and optionally `stream()`). It demonstrates **question mode**, **email mode**, and **HITL resume** (notify: respond/ignore; tool approval: approve or decline send_email/schedule_meeting).

**Prerequisites:** Agent implemented; venv activated and kernel set to that venv; `.env` (or env vars) for `OPENAI_API_KEY`. Those are optionally: LangSmith, Gmail, and Postgres.

## 1. Setup

Add repo root to `sys.path` and load `.env` so env vars are set before any LangGraph/LangChain imports.

In [10]:
import sys
from pathlib import Path

# Assume notebook is run from repo root (or notebooks/); repo root = parent of notebooks/
repo_root = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dotenv import load_dotenv
load_dotenv(repo_root / ".env")

True

## 2. Imports

Import the compiled graph and `Command` from `langgraph.types` for HITL resume.

In [11]:
from langchain_core.messages import HumanMessage
from langgraph.types import Command

from email_assistant.email_assistant_hitl_memory_gmail import build_email_assistant_graph
from langgraph.checkpoint.memory import MemorySaver

## 3. Config

Set `thread_id` and `user_id`; build config for invoke.

In [12]:
thread_id = "notebook-thread-1"
user_id = "notebook-user"
config = {"configurable": {"thread_id": thread_id, "user_id": user_id}}

## 4. Run question mode

Invoke with `user_message` only (no email). Print last message.

In [13]:
graph = build_email_assistant_graph(checkpointer=MemorySaver())
result = graph.invoke({"user_message": "What can you help me with?"}, config=config)
messages = result.get("messages", [])
if messages:
    print(messages[-1].content if hasattr(messages[-1], "content") else messages[-1])
else:
    print("(no messages)")

I can assist you with a variety of tasks, including:

1. **Email Management**: I can help you send, reply to, and check your emails.
2. **Calendar Management**: I can check your calendar for upcoming events and schedule new meetings.
3. **General Information and Questions**: I can provide information on a wide range of topics and answer questions.
4. **Task Automation**: I can help automate certain routine tasks to make your workflow more efficient.

Please let me know how I can assist you today!


## 5. Run email mode

Invoke with `email_input` (mock or real). Print classification and state.

In [14]:
email_input = {
    "from": "colleague@example.com",
    "to": "you@example.com",
    "subject": "FYI deploy completed",
    "body": "The deploy finished successfully. No action needed."
}
result = graph.invoke({"email_input": email_input}, config=config)
print("Classification:", result.get("classification_decision", "(none)"))
if result.get("__interrupt__"):
    print("Interrupt (notify path):", result["__interrupt__"])

Classification: (none)
Interrupt (notify path): [Interrupt(value={'message': 'This email was classified as **notify** (FYI). Should the assistant respond or ignore?', 'options': ['respond', 'ignore']}, id='cf924f1636a781b2f2862143ece22cc4')]


## 6. Handle HITL

After invoke, if `result.get("__interrupt__")` is set, the graph is paused. **The cell will wait for your input:** for notify interrupts type `r` (respond) or `i` (ignore); for tool approval type `y` or `n`. Resume uses `Command(resume=<choice>)` with the same `thread_id`.

In [15]:
while result.get("__interrupt__"):
    raw = result["__interrupt__"]
    # LangGraph may return a list of Interrupt objects; normalize to a dict (e.g. first item's .value)
    if isinstance(raw, list) and raw:
        first = raw[0]
        payload = getattr(first, "value", first) if not isinstance(first, dict) else first
    else:
        payload = raw if isinstance(raw, dict) else {}
    msg = payload.get("message", raw) if isinstance(payload, dict) else raw
    print("Paused:", msg)
    if isinstance(payload, dict) and "tool_calls" in payload:
        raw_choice = input("Approve tool execution? (y/n): ").strip().lower()
        choice = raw_choice != "n"
    else:
        raw_choice = input("(r)espond or (i)gnore? ").strip().lower()
        choice = "respond" if raw_choice.startswith("r") else "ignore"
    result = graph.invoke(Command(resume=choice), config=config)

messages = result.get("messages", [])
if messages:
    print("Last:", messages[-1].content[:200] if hasattr(messages[-1], "content") else messages[-1])

Paused: This email was classified as **notify** (FYI). Should the assistant respond or ignore?
Last: I can assist you with a range of tasks, including:

1. **Email Management**: Sending, retrieving, and replying to emails.
2. **Calendar Management**: Checking your calendar for events, scheduling new 


## 7. Optional: stream or get_state

Use `graph.stream(..., config=config, stream_mode="values")` for chunks, or `graph.get_state(config)` to inspect the latest checkpoint.

In [16]:
# Example: stream values
# for chunk in graph.stream({"user_message": "Hello"}, config=config, stream_mode="values"):
#     print(chunk)

# Example: get state
state = graph.get_state(config)
print("State keys:", list(state.values.keys()) if hasattr(state, "values") else state)

State keys: ['messages', 'email_input', 'classification_decision', '_notify_choice', 'user_message']
